# Neuron-seed recovery rate on the REAL datasets (thousands of organisms)

How often does the **mechanistic** neuron-seed method recover a memorized secret with *no* search,
measured across thousands of organisms per dataset?

* **48-bit 1-layer**  `g(x) = W2 . relu(W1.x + b1) + b2`           (dataset/chunk*.pt)
* **34-bit 3-layer**  `relu^2(rmsnorm(W_i.h + b_i, g_i)) x3 -> Wo`  (gendatasetlpe34bit...)
* **64-bit 3-layer**  same 3-layer form, H=128                      (dataset64/shard*.pt)

A neuron seed comes from a composed-product detector — `W1`, `W2.W1`, `W3.W2.W1`. For each detector row
we take **every threshold cutoff** that yields a distinct string (the `D+1` "top-k coordinates = 1" strings
from `00..0` to `11..1`). We report recovery /16 **broken down by product** (`p1`=W1, `p2`=W2.W1,
`p3`=W3.W2.W1) and **combined** (`comb` = the per-organism *union* of secrets recovered by any product), for:

* **`neuron`** — the candidate strings used directly, **no optimization** (run on all `N_ORGS`).
* **`neuron_hillclimb`** — the same seeds polished with ICM + 2-bit (run on the `N_HILL` subset; expensive).

`RUN_SEARCH` (rand_ICM / grad / GCG) and `RUN_ARC` are off by default — they're already ~0 on the 3-layer
nets and only slow the big run down. Recovery is checked directly against the known secrets (no brute force).

In [1]:
import os, glob, time, math
import torch, torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
print("device:", device, "| GPUs:", torch.cuda.device_count())

# ---- how many organisms (loaded across as many shards as needed) ----
N_ORGS = 2000          # organisms tested per dataset; PURE neuron (cheap) runs on all of them
N_HILL = 100           # subset that ALSO gets neuron_hillclimb (ICM/2bit polish) -- the EXPENSIVE part (~1/4 of N_ORGS);
                       #   lower further or set RUN_HILLCLIMB=False if the 64-bit run is too slow

# ---- which extras to run (off by default for the big neuron-focused run) ----
RUN_HILLCLIMB = True   # neuron_hillclimb: polish the neuron seeds with ICM + 2-bit
RUN_SEARCH    = False  # rand_ICM / rand_2bit / grad_2bit / gcg (already ~0 on 3-layer; expensive)
RUN_ARC       = False  # ARC estimators (ITGIS/MHIS/GLD)
ARC_ORGS      = 8      # ARC is slow -> only the first few orgs when RUN_ARC

# datasets: name, arch, bit-width D, candidate paths, hidden H (for reporting)
DATASETS = [
    dict(name="48bit-1L", arch="1l", D=48, H=128,
         paths=["dataset", "../dataset", "/kaggle/input/notebooks/emanuelruzak/lpe9datagen48bit-16strings-ipynb", "dataset48"]),
    dict(name="34bit-3L", arch="3l", D=34, H=64,
         paths=["/kaggle/input/notebooks/emanuelruzak/gendatasetlpe34bit3layerrelu2", "dataset34", "../dataset34"]),
    dict(name="64bit-3L", arch="3l", D=64, H=128,
         paths=["/kaggle/input/notebooks/emanuelruzak/gendatasetlpe64bit3layerrelu2", "3layerstuff/dataset64", "../3layerstuff/dataset64"]),
]

# ---- search / polish config ----
READER_FLOPS  = 3.0e9
SEARCH_FLOP_BUDGET = 40 * READER_FLOPS
R_MAX        = 1 << 16
S_SWEEP      = 16
ESCAPE_ROUNDS = 3
K2BIT        = 64
GRAD_RESTARTS = 256
GRAD_STEPS    = 30
GRAD_LR       = 0.1
GCG_RESTARTS = 128
GCG_STEPS    = 150
GCG_TOPK     = 16
# ARC estimators (vanilla)
EST_TEMP, EST_SAMPLES = 1.0, 8192
ITGIS_ITERS, ITGIS_BATCH = 10, 512
MHIS_CHAINS, MHIS_STEPS, MHIS_BURN = 256, 200, 100

device: cuda | GPUs: 2


In [2]:
# ---- architecture-correct forward g(x) (logit) for the loaded nets ----
def rmsnorm(z, g, eps=1e-6):
    return z * torch.rsqrt(z.pow(2).mean(-1, keepdim=True) + eps) * g

FWD = [0]
def g1(w, X):                                     # X (Nc, D) -> (Nc,) logits
    FWD[0] += X.shape[0]
    if w["arch"] == "1l":
        h = F.relu(X @ w["W1"].t() + w["b1"])
        return (h @ w["W2"].t() + w["b2"]).squeeze(-1)
    Ws, bs, gs = w["Ws"], w["bs"], w["gs"]        # 3-layer relu^2 + rmsnorm
    h = X
    for i in range(3):
        h = F.relu(rmsnorm(h @ Ws[i].t() + bs[i], gs[i])) ** 2
    return (h @ Ws[3].t() + bs[3]).squeeze(-1)

def fwd_flops_net(w):
    if w["arch"] == "1l":
        H, Dn = w["W1"].shape; return 2 * H * Dn + 2 * H + 2 * H
    Ws = w["Ws"]; H, Dn = Ws[0].shape
    return 2 * H * Dn + 2 * (2 * H * H) + 2 * H

# ---- neuron detectors: the composed products, kept SEPARATE per layer ----
def neuron_products(w):                           # list of (H, D) detectors: [W1] or [W1, W2@W1, W3@W2@W1]
    if w["arch"] == "1l":
        return [w["W1"]]
    prods, M = [], None
    for i in range(3):
        M = w["Ws"][i] if M is None else w["Ws"][i] @ M        # W1, W2@W1, W3@W2@W1
        prods.append(M)
    return prods

def threshold_seeds(M):                           # (V,D) -> ALL distinct cutoff strings per row (0..0 -> 1..1), deduped
    V, Dd = M.shape                               # seed_i = 1[m_i > tau]; sweeping tau gives the D+1 "top-k bits on" strings
    order = M.argsort(dim=1, descending=True)      # positions sorted by weight, largest first
    rank = torch.empty_like(order)
    rank.scatter_(1, order, torch.arange(Dd, device=M.device).expand(V, Dd))
    ks = torch.arange(Dd + 1, device=M.device).view(1, Dd + 1, 1)
    seeds = (rank.unsqueeze(1) < ks).float().reshape(V * (Dd + 1), Dd)   # top-k bits = 1 for k = 0..D
    return torch.unique(seeds, dim=0)

# ---- generic search battery (same as the sweep), operating on global D ----
PAIRS = {}
@torch.no_grad()
def icm_sweeps(w, X, S):
    for _ in range(S):
        for i in torch.randperm(D):
            X0 = X.clone(); X0[:, i] = 0; X1 = X.clone(); X1[:, i] = 1
            X[:, i] = (g1(w, X1) > g1(w, X0)).float()
    return X

@torch.no_grad()
def best_2bit(w, X):
    key = (X.device, D)
    if key not in PAIRS:
        PAIRS[key] = torch.tensor([(i, j) for i in range(D) for j in range(i + 1, D)], device=X.device)
    best = g1(w, X).clone(); bestX = X.clone()
    for p in PAIRS[key]:
        Xc = X.clone(); Xc[:, p[0]] = 1 - Xc[:, p[0]]; Xc[:, p[1]] = 1 - Xc[:, p[1]]
        gc = g1(w, Xc); imp = gc > best
        best = torch.where(imp, gc, best); bestX[imp] = Xc[imp]
    return bestX

@torch.no_grad()
def polish(w, seeds, S, two_bit, k2bit):
    if seeds.shape[0] == 0: return seeds
    X = icm_sweeps(w, seeds, S)
    if two_bit:
        for _ in range(ESCAPE_ROUNDS):
            top = g1(w, X).argsort(descending=True)[:min(k2bit, X.shape[0])]
            Xt = icm_sweeps(w, best_2bit(w, X[top].clone()), S)
            X = X.clone(); X[top] = Xt
    return X

def grad_logit(w, X):
    with torch.enable_grad():
        Xr = X.detach().clone().requires_grad_(True)
        gr, = torch.autograd.grad(g1(w, Xr).sum(), Xr)
    return gr.detach()

def grad_seeds(w, R, steps, lr, dev):
    with torch.enable_grad():
        x = torch.rand(R, D, device=dev)
        for _ in range(steps):
            x.requires_grad_(True)
            grad, = torch.autograd.grad(g1(w, x).sum(), x)
            x = (x.detach() + lr * grad.sign()).clamp(0, 1)
    return (x > 0.5).float()

@torch.no_grad()
def gcg_climb(w, R, n_steps, top_k, dev):
    X = (torch.rand(R, D, device=dev) > 0.5).float()
    cur = g1(w, X); rows = torch.arange(R, device=dev); k = min(top_k, D)
    for _ in range(n_steps):
        gain = grad_logit(w, X) * (1.0 - 2.0 * X)
        cand = gain.topk(k, dim=1).indices
        Xrep = X.unsqueeze(1).expand(-1, k, -1).clone()
        rr = rows.view(-1, 1).expand(-1, k); cc = torch.arange(k, device=dev).view(1, -1).expand(R, -1)
        Xrep[rr, cc, cand] = 1.0 - Xrep[rr, cc, cand]
        cl = g1(w, Xrep.reshape(-1, D)).reshape(R, k)
        bestval, bestj = cl.max(dim=1); improve = bestval > cur
        bestpos = cand[rows, bestj]
        X[rows[improve], bestpos[improve]] = 1.0 - X[rows[improve], bestpos[improve]]
        cur = torch.where(improve, bestval, cur)
        if not improve.any(): break
    return X

@torch.no_grad()
def recov_mask(X, secrets):                       # bool (16,): which of the 16 secrets appear among candidates X
    if X.shape[0] == 0:
        return torch.zeros(secrets.shape[0], dtype=torch.bool, device=secrets.device)
    C = torch.unique((X > 0.5).float(), dim=0)
    sh = torch.arange(D, dtype=torch.int64, device=X.device)
    ci = (C.to(torch.int64) << sh).sum(-1); ti = (secrets.to(torch.int64) << sh).sum(-1)
    return torch.isin(ti, ci)

def recov(X, secrets):                            # count version
    return int(recov_mask(X, secrets).sum())

# ---- neuron evaluation: per matrix product (p1=W1, p2=W2.W1, p3=W3.W2.W1) + combined (union of recovered secrets) ----
def neuron_eval(w, secrets, do_hillclimb):
    prods = neuron_products(w)                    # 1 (1-layer) or 3 (3-layer) detector matrices
    pure, hc = [], []
    for M in prods:
        seeds = threshold_seeds(M)                # all-cutoff strings for THIS product only
        pure.append(recov_mask(seeds, secrets))                                          # pure: no optimization
        if do_hillclimb:
            hc.append(recov_mask(polish(w, seeds.clone(), S_SWEEP, True, K2BIT), secrets))  # seeds + ICM/2bit polish
    out = {}
    for j in range(3):                            # p2/p3 are NaN for the 1-layer net (only W1 exists)
        out[f"neuron_p{j+1}"] = float(pure[j].sum()) if j < len(prods) else float("nan")
    out["neuron_comb"] = float(torch.stack(pure).any(0).sum())                           # union over products
    if do_hillclimb:
        for j in range(3):
            out[f"neuronhc_p{j+1}"] = float(hc[j].sum()) if j < len(prods) else float("nan")
        out["neuronhc_comb"] = float(torch.stack(hc).any(0).sum())
    return out

# ---- the other (input-optimization) methods, for the optional sanity check ----
def search_methods(w, secrets, R1, dev):
    rand = (torch.rand(R1, D, device=dev) > 0.5).float()
    return {
        "rand_ICM":  recov(polish(w, rand.clone(), S_SWEEP, False, K2BIT), secrets),
        "rand_2bit": recov(polish(w, rand.clone(), S_SWEEP, True,  K2BIT), secrets),
        "grad_2bit": recov(polish(w, grad_seeds(w, GRAD_RESTARTS, GRAD_STEPS, GRAD_LR, dev), S_SWEEP, True, K2BIT), secrets),
        "gcg":       recov(gcg_climb(w, GCG_RESTARTS, GCG_STEPS, GCG_TOPK, dev), secrets),
    }

In [3]:
# ---- ARC estimators (no ground truth here -- there is no brute force) ----
def grad_g(w, X):
    with torch.enable_grad():
        Xr = X.detach().clone().requires_grad_(True)
        gv = g1(w, Xr); grad, = torch.autograd.grad(gv.sum(), Xr)
    return gv.detach(), grad.detach()

@torch.no_grad()
def gld(w, thr, dev):
    x = (torch.rand(EST_SAMPLES, D, device=dev) < 0.5).float()
    delta = g1(w, x) - thr
    mu, sd = delta.mean().item(), delta.std().clamp_min(1e-6).item()
    return 0.5 * math.erfc(-(mu / sd) / math.sqrt(2))

def itgis(w, thr, dev):
    rho = torch.full((D,), 0.5, device=dev)
    for _ in range(ITGIS_ITERS):
        x = (torch.rand(ITGIS_BATCH, D, device=dev) < rho).float()
        _, grad = grad_g(w, x)
        rho = torch.sigmoid(grad.mean(0) / EST_TEMP).clamp(1e-4, 1 - 1e-4)
    x = (torch.rand(EST_SAMPLES, D, device=dev) < rho).float()
    gx = g1(w, x)
    logq = (x * torch.log(rho) + (1 - x) * torch.log(1 - rho)).sum(1)
    w_is = torch.exp(D * math.log(0.5) - logq); acc = (gx >= thr).float()
    return (w_is * acc).mean().item(), x[acc.bool()]

def mhis(w, thr, dev):
    T = EST_TEMP
    xu = (torch.rand(EST_SAMPLES, D, device=dev) < 0.5).float()
    gu = g1(w, xu); gmax = gu.max(); Z = torch.exp((gu - gmax) / T).mean()
    x = (torch.rand(MHIS_CHAINS, D, device=dev) < 0.5).float(); samples = []
    for step in range(MHIS_STEPS):
        i = torch.randint(0, D, (MHIS_CHAINS, 1), device=dev)
        gx, grad = grad_g(w, x)
        gi = grad.gather(1, i).squeeze(1); xi = x.gather(1, i).squeeze(1)
        p1 = torch.sigmoid(gi / T)
        newv = (torch.rand(MHIS_CHAINS, device=dev) < p1).float()
        xprop = x.clone(); xprop.scatter_(1, i, newv[:, None])
        gp, gradp = grad_g(w, xprop)
        p1r = torch.sigmoid(gradp.gather(1, i).squeeze(1) / T)
        pf = torch.where(newv == 1, p1, 1 - p1); pr = torch.where(xi == 1, p1r, 1 - p1r)
        log_acc = (gp - gx) / T + torch.log(pr + 1e-12) - torch.log(pf + 1e-12)
        take = (torch.rand(MHIS_CHAINS, device=dev) < torch.exp(log_acc.clamp(max=0)))
        x = torch.where(take[:, None], xprop, x)
        if step >= MHIS_BURN: samples.append(x.clone())
    s = torch.cat(samples, 0); gs = g1(w, s); acc = (gs >= thr).float()
    est = (Z * (torch.exp((gmax - gs) / T) * acc).mean()).item()
    return est, s[acc.bool()]

In [4]:
# ---- dataset loaders (walk shards until n organisms are collected) ----
def find_dir(paths, pat):
    for p in paths:
        if os.path.isdir(p) and glob.glob(os.path.join(p, "**", pat), recursive=True):
            return p
    return None

def load_1l(path, n, dev):
    out = []
    for f in sorted(glob.glob(os.path.join(path, "**", "*.pt"), recursive=True)):
        if len(out) >= n: break
        b = torch.load(f, map_location="cpu", weights_only=False)
        for i in range(b["W1"].shape[0]):
            if len(out) >= n: break
            w = dict(arch="1l", W1=b["W1"][i].float().to(dev), b1=b["b1"][i].float().to(dev),
                     W2=b["W2"][i].float().to(dev), b2=b["b2"][i].float().to(dev))
            secrets = b["templates"][i, :, 1:].float().to(dev)      # col0 is a BOS marker; 48 bits follow
            out.append((w, secrets))
    return out

def load_3l(path, n, dev):
    out = []
    for f in sorted(glob.glob(os.path.join(path, "**", "shard_*.pt"), recursive=True)):
        if len(out) >= n: break
        b = torch.load(f, map_location="cpu", weights_only=False)
        for i in range(b["targets"].shape[0]):
            if len(out) >= n: break
            w = dict(arch="3l",
                     Ws=[b["Ws"][j][i].float().to(dev) for j in range(4)],
                     bs=[b["bs"][j][i].float().to(dev) for j in range(4)],
                     gs=[b["gs"][j][i].float().to(dev) for j in range(3)])
            out.append((w, b["targets"][i].float().to(dev)))
    return out

In [5]:
# ---- run the neuron-rate experiment on each dataset (thousands of organisms) ----
NEU_PURE = ["neuron_p1", "neuron_p2", "neuron_p3", "neuron_comb"]      # p1=W1, p2=W2.W1, p3=W3.W2.W1, comb=union
NEU_HC   = ["neuronhc_p1", "neuronhc_p2", "neuronhc_p3", "neuronhc_comb"]
SRCH     = ["rand_ICM", "rand_2bit", "grad_2bit", "gcg"]

rows = []
for ds in DATASETS:
    pat = "*.pt" if ds["arch"] == "1l" else "shard_*.pt"
    path = find_dir(ds["paths"], pat)
    if path is None:
        print(f"[{ds['name']}] no data found in {ds['paths']} -- skipping\n"); continue
    D = ds["D"]                                                        # <-- global D for the battery
    orgs = (load_1l if ds["arch"] == "1l" else load_3l)(path, N_ORGS, device)
    ff = fwd_flops_net(orgs[0][0])
    R1 = int(min(R_MAX, max(64, (SEARCH_FLOP_BUDGET / ff) / (2 * D * S_SWEEP * 4)))) if RUN_SEARCH else 0
    nh = min(N_HILL, len(orgs)) if RUN_HILLCLIMB else 0
    print(f"=== {ds['name']}  D={D} H={ds['H']}  ({len(orgs)} orgs from {path}) | hillclimb on {nh}"
          + (f" | search R1={R1}" if RUN_SEARCH else "") + " ===")
    agg, cnt, mp_all = {}, {}, []
    est = {"gld": [], "itgis": [], "mhis": []}; rec = {"itgis": [], "mhis": []}
    def add(k, v, c=1):                                                # running sum/count, skipping NaN (absent product)
        if v != v: return
        agg[k] = agg.get(k, 0.0) + v; cnt[k] = cnt.get(k, 0) + c
    t0 = time.time()
    for idx, (w, secrets) in enumerate(orgs):
        with torch.no_grad():
            sl = g1(w, secrets); mp_all.append(torch.sigmoid(sl).min().item()); thr = sl.min().item()
        ne = neuron_eval(w, secrets, do_hillclimb=(idx < nh))
        for k in NEU_PURE: add(k, ne[k])
        if idx < nh:
            for k in NEU_HC: add(k, ne[k])
        if RUN_SEARCH:
            for k, v in search_methods(w, secrets, R1, device).items(): add(k, v)
        if RUN_ARC and idx < ARC_ORGS:
            est["gld"].append(gld(w, thr, device))
            e_i, ci = itgis(w, thr, device); est["itgis"].append(e_i); rec["itgis"].append(recov(ci, secrets) if ci.numel() else 0)
            e_m, cm = mhis(w, thr, device);  est["mhis"].append(e_m); rec["mhis"].append(recov(cm, secrets) if cm.numel() else 0)
        if (idx + 1) % 200 == 0:
            print(f"   ... {idx+1}/{len(orgs)} | neuron_comb {agg['neuron_comb']/cnt['neuron_comb']:.3f}/16 | {time.time()-t0:.0f}s")
    M = {k: agg[k] / cnt[k] for k in agg}
    row = dict(name=ds["name"], D=D, H=ds["H"], n=len(orgs), n_hill=nh, min_pos=sum(mp_all) / len(mp_all), **M)
    if RUN_ARC and est["gld"]:
        row.update(gld=sum(est["gld"]) / len(est["gld"]), itgis=sum(est["itgis"]) / len(est["itgis"]),
                   mhis=sum(est["mhis"]) / len(est["mhis"]),
                   rec_itgis=sum(rec["itgis"]) / len(rec["itgis"]), rec_mhis=sum(rec["mhis"]) / len(rec["mhis"]))
    rows.append(row)
    show = lambda ks: " ".join(f"{k.split('_')[-1]}:{M[k]:.3f}" for k in ks if k in M)
    print(f"   min_pos {row['min_pos']:.4f} | NEURON pure {show(NEU_PURE)}"
          + (f" | NEURON hillclimb {show(NEU_HC)}" if RUN_HILLCLIMB else "")
          + (f" | SEARCH {show(SRCH)}" if RUN_SEARCH else "")
          + f" | {time.time()-t0:.0f}s\n")
print("done.")

=== 48bit-1L  D=48 H=128  (2000 orgs from /kaggle/input/notebooks/emanuelruzak/lpe9datagen48bit-16strings-ipynb) | hillclimb on 100 ===
   ... 200/2000 | neuron_comb 2.140/16 | 237s
   ... 400/2000 | neuron_comb 2.183/16 | 238s
   ... 600/2000 | neuron_comb 2.133/16 | 238s
   ... 800/2000 | neuron_comb 2.160/16 | 239s
   ... 1000/2000 | neuron_comb 2.195/16 | 240s
   ... 1200/2000 | neuron_comb 2.184/16 | 240s
   ... 1400/2000 | neuron_comb 2.189/16 | 241s
   ... 1600/2000 | neuron_comb 2.195/16 | 242s
   ... 1800/2000 | neuron_comb 2.169/16 | 242s
   ... 2000/2000 | neuron_comb 2.171/16 | 243s
   min_pos 0.9999 | NEURON pure p1:2.171 comb:2.171 | NEURON hillclimb p1:15.740 comb:15.740 | 243s

=== 34bit-3L  D=34 H=64  (2000 orgs from /kaggle/input/notebooks/emanuelruzak/gendatasetlpe34bit3layerrelu2) | hillclimb on 100 ===
   ... 200/2000 | neuron_comb 0.010/16 | 1112s
   ... 400/2000 | neuron_comb 0.007/16 | 1114s
   ... 600/2000 | neuron_comb 0.005/16 | 1115s
   ... 800/2000 | neuron

In [6]:
# ---- results: neuron recovery rate per matrix product + combined ----
import pandas as pd
df = pd.DataFrame(rows)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 60)

pure_cols = [c for c in NEU_PURE if c in df.columns]
print("=== NEURON (pure, no optimization): mean recovery /16 per composed product + combined ===")
print(df[["name", "D", "H", "n", "min_pos"] + pure_cols].to_string(index=False))
print("  p1=sign-thresholds of W1 | p2=W2.W1 | p3=W3.W2.W1 | comb = secrets recovered by ANY product (per-organism union)")

hc_cols = [c for c in NEU_HC if c in df.columns]
if hc_cols:
    print("\n=== NEURON_HILLCLIMB (seeds + ICM/2bit polish): mean recovery /16 per product + combined ===")
    print(df[["name", "D", "H", "n_hill"] + hc_cols].to_string(index=False))

if "neuron_comb" in df.columns:
    print("\n=== combined recovery as % of the 16 secrets ===")
    pct = df[["name"]].copy()
    pct["neuron_comb_%"] = (df["neuron_comb"] / 16 * 100).round(2)
    if "neuronhc_comb" in df.columns:
        pct["neuronhc_comb_%"] = (df["neuronhc_comb"] / 16 * 100).round(2)
    print(pct.to_string(index=False))

srch_cols = [c for c in SRCH if c in df.columns]
if srch_cols:
    print("\n=== input-optimization battery (mean /16), if RUN_SEARCH ===")
    print(df[["name"] + srch_cols].to_string(index=False))

=== NEURON (pure, no optimization): mean recovery /16 per composed product + combined ===
    name  D   H    n  min_pos  neuron_p1  neuron_p2  neuron_p3  neuron_comb
48bit-1L 48 128 2000 0.999931     2.1715        NaN        NaN       2.1715
34bit-3L 34  64 2000 0.999992     0.0055        0.0        0.0       0.0055
64bit-3L 64 128 2000 0.999955     1.5080        0.0        0.0       1.5080
  p1=sign-thresholds of W1 | p2=W2.W1 | p3=W3.W2.W1 | comb = secrets recovered by ANY product (per-organism union)

=== NEURON_HILLCLIMB (seeds + ICM/2bit polish): mean recovery /16 per product + combined ===
    name  D   H  n_hill  neuronhc_p1  neuronhc_p2  neuronhc_p3  neuronhc_comb
48bit-1L 48 128     100        15.74          NaN          NaN          15.74
34bit-3L 34  64     100         0.07         0.02         0.01           0.09
64bit-3L 64 128     100         1.68         0.00         0.00           1.68

=== combined recovery as % of the 16 secrets ===
    name  neuron_comb_%  neuronhc_c